# FMP Data in Quant Warehouse

This notebook is a read-first tour of the FMP-backed data currently exposed by `quant-warehouse`. It shows the major data families and the warehouse APIs used to access them.

Set `RUN_REFRESH = True` only when you want the notebook to download missing FMP data into your local warehouse.

In [ ]:
from __future__ import annotations

from dataclasses import asdict

import pandas as pd

from quant_warehouse.warehouse import Warehouse
from quant_warehouse.warehouse.storage import provider_library
from quant_warehouse.platforms.data_providers.fmp.sections import (
    FMP_HISTORICAL_EQUITY_SECTIONS,
    FMP_EXTENDED_EQUITY_SECTIONS,
    FMP_HISTORICAL_ETF_SECTIONS,
)
from quant_warehouse.warehouse.sections import (
    EQUITY_CALENDAR_SECTIONS,
    MARKET_PRICE_SECTIONS,
    MACRO_ECONOMIC_SECTION,
    MACRO_TREASURY_SECTION,
    MACRO_YIELD_CURVE_SECTION,
    CRYPTO_PRICES_LIBRARY,
    CURRENCY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    FUND_PRICES_LIBRARY,
    INDEX_PRICES_LIBRARY,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_TREASURY_LIBRARY,
    MACRO_YIELD_CURVE_LIBRARY,
    EQUITY_PRICES_LIBRARY,
    fundamental_library,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

warehouse = Warehouse()

PROVIDER = "fmp"
RUN_REFRESH = False

EQUITY_SYMBOL = "AAPL"
ETF_SYMBOL = "SPY"
MUTUAL_FUND_SYMBOL = "VFIAX"
CRYPTO_SYMBOL = "BTCUSD"
CURRENCY_SYMBOL = "EURUSD"
INDEX_SYMBOL = "^GSPC"

## Helpers

In [ ]:
def preview(frame: pd.DataFrame, rows: int = 5) -> pd.DataFrame:
    if frame is None or frame.empty:
        return pd.DataFrame()
    return frame.tail(rows)


def state(symbol: str, section: str, provider: str = PROVIDER) -> dict[str, object] | None:
    item = warehouse.catalog.get(symbol=symbol, section=section, provider=provider)
    return asdict(item) if item is not None else None


def state_table(symbol: str, sections: list[str] | tuple[str, ...], provider: str = PROVIDER) -> pd.DataFrame:
    rows = [state(symbol, section, provider=provider) for section in sections]
    rows = [row for row in rows if row is not None]
    table = pd.DataFrame(rows)
    if not table.empty and "columns_present" in table.columns:
        table.insert(
            table.columns.get_loc("columns_present"),
            "column_count",
            table["columns_present"].map(len),
        )
    return table


def read_fundamental(symbol: str, section: str, rows: int = 5) -> pd.DataFrame:
    return preview(warehouse.read_fundamentals(symbol, section=section, provider=PROVIDER), rows=rows)

## FMP Storage Coverage by Symbol

In [ ]:
equity_sections = ["prices", "profile", *FMP_HISTORICAL_EQUITY_SECTIONS, *FMP_EXTENDED_EQUITY_SECTIONS]
state_table(EQUITY_SYMBOL, equity_sections)

## Equity Prices

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_prices(EQUITY_SYMBOL, providers=[PROVIDER])

equity_prices = warehouse.read_prices(EQUITY_SYMBOL, provider=PROVIDER)
preview(equity_prices)

## Equity Profile

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_profile(EQUITY_SYMBOL, provider=PROVIDER)

profile = warehouse.read_profile(EQUITY_SYMBOL, provider=PROVIDER)
asdict(profile) if profile is not None else None

## Equity Fundamentals

These sections map to FMP/OpenBB equity fundamental data. They are intentionally equity-specific; they should not be used as the default path for ETF or mutual fund symbols.

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["income", "balance", "cash", "metrics", "ratios", "dividends", "historical_splits"],
        providers=[PROVIDER],
        period="quarter",
    )

state_table(EQUITY_SYMBOL, FMP_HISTORICAL_EQUITY_SECTIONS)

In [ ]:
read_fundamental(EQUITY_SYMBOL, "income")

In [ ]:
read_fundamental(EQUITY_SYMBOL, "ratios")

In [ ]:
read_fundamental(EQUITY_SYMBOL, "dividends")

## Extended FMP Equity Data

These are still FMP-derived equity sections, but they represent event, estimate, ownership, filing, or snapshot-style datasets rather than standard financial statements.

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_fundamentals(
        EQUITY_SYMBOL,
        sections=["historical_market_cap", "filings", "estimates_price_target", "ownership_insider_trading"],
        providers=[PROVIDER],
    )

state_table(EQUITY_SYMBOL, FMP_EXTENDED_EQUITY_SECTIONS)

In [ ]:
read_fundamental(EQUITY_SYMBOL, "historical_market_cap")

In [ ]:
read_fundamental(EQUITY_SYMBOL, "filings")

## ETF Prices and ETF-Specific Data

ETF data is separated from equity fundamentals. Use `warehouse.etf` or ETF sections through `warehouse.read_fundamentals`.

In [ ]:
if RUN_REFRESH:
    warehouse.etf.refresh_prices(ETF_SYMBOL, providers=[PROVIDER])
    warehouse.etf.refresh_profile(ETF_SYMBOL, provider=PROVIDER)
    warehouse.etf.refresh_fundamentals(ETF_SYMBOL, sections=["etf_holdings", "etf_sectors"], providers=[PROVIDER])

preview(warehouse.etf.read_prices(ETF_SYMBOL, provider=PROVIDER))

In [ ]:
state_table(ETF_SYMBOL, ["etf_prices", "etf_profile", *FMP_HISTORICAL_ETF_SECTIONS])

In [ ]:
read_fundamental(ETF_SYMBOL, "etf_holdings")

## Mutual Fund Prices

Mutual funds should be treated as their own instrument type. Prices can be read through the price API, but equity fundamentals are not a valid default assumption for mutual fund symbols.

In [ ]:
mutual_fund_prices = warehouse.read_prices(MUTUAL_FUND_SYMBOL, provider=PROVIDER)
preview(mutual_fund_prices)

In [ ]:
state_table(MUTUAL_FUND_SYMBOL, ["fund_prices", "prices", "income", "balance", "cash"])

## Crypto, Currency, and Index Prices

FMP market price routes are stored outside equity/ETF/fund price sections.

In [ ]:
if RUN_REFRESH:
    warehouse.market_prices.refresh(CRYPTO_SYMBOL, section="crypto_prices", provider=PROVIDER)
    warehouse.market_prices.refresh(CURRENCY_SYMBOL, section="currency_prices", provider=PROVIDER)
    warehouse.market_prices.refresh(INDEX_SYMBOL, section="index_prices", provider=PROVIDER)

market_examples = {
    "crypto": warehouse.market_prices.read(CRYPTO_SYMBOL, section="crypto_prices", provider=PROVIDER),
    "currency": warehouse.market_prices.read(CURRENCY_SYMBOL, section="currency_prices", provider=PROVIDER),
    "index": warehouse.market_prices.read(INDEX_SYMBOL, section="index_prices", provider=PROVIDER),
}

{name: preview(frame) for name, frame in market_examples.items()}

In [ ]:
pd.DataFrame(
    [
        state(CRYPTO_SYMBOL, "crypto_prices"),
        state(CURRENCY_SYMBOL, "currency_prices"),
        state(INDEX_SYMBOL, "index_prices"),
    ]
).dropna(how="all")

## Macro Data

FMP macro data is stored as economic series, treasury rates, yield curve history, calendars, and risk premium snapshots.

In [ ]:
if RUN_REFRESH:
    warehouse.refresh_macro(
        economic_series=["GDP", "CPI"],
        include_treasury_rates=True,
        include_yield_curve=True,
        provider=PROVIDER,
    )

macro_states = [
    state("GDP", MACRO_ECONOMIC_SECTION),
    state("CPI", MACRO_ECONOMIC_SECTION),
    state("TREASURY_CURVE", MACRO_TREASURY_SECTION),
    state("YIELD_CURVE", MACRO_YIELD_CURVE_SECTION),
]
pd.DataFrame([row for row in macro_states if row is not None])

In [ ]:
macro_panel = warehouse.read_macro_panel(["GDP", "CPI"], provider=PROVIDER)
preview(macro_panel)

## Equity Calendars

Equity calendars are market-wide FMP panels, not symbol-level fundamentals.

In [ ]:
if RUN_REFRESH:
    warehouse.equity_calendar.refresh_all(provider=PROVIDER, start_date="2024-01-01")

calendar_state = state_table("EQUITY_CALENDAR", EQUITY_CALENDAR_SECTIONS)
calendar_state

In [ ]:
preview(warehouse.equity_calendar.read("equity_calendar_earnings", provider=PROVIDER))

## Provider-Separated Libraries

The warehouse stores provider-backed sections in provider-specific Arctic libraries. This lets FMP, yfinance, and ThetaData evolve independently and reduces the chance that heavy writes in one provider interfere with another provider's data.

In [ ]:
expected_base_libraries = [
    EQUITY_PRICES_LIBRARY,
    ETF_PRICES_LIBRARY,
    FUND_PRICES_LIBRARY,
    CRYPTO_PRICES_LIBRARY,
    CURRENCY_PRICES_LIBRARY,
    INDEX_PRICES_LIBRARY,
    MACRO_ECONOMIC_LIBRARY,
    MACRO_TREASURY_LIBRARY,
    MACRO_YIELD_CURVE_LIBRARY,
    *[fundamental_library(section) for section in FMP_HISTORICAL_EQUITY_SECTIONS],
    *[fundamental_library(section) for section in FMP_EXTENDED_EQUITY_SECTIONS],
    *[fundamental_library(section) for section in FMP_HISTORICAL_ETF_SECTIONS],
    *[fundamental_library(section) for section in EQUITY_CALENDAR_SECTIONS],
]

pd.DataFrame(
    {
        "base_library": sorted(set(expected_base_libraries)),
        "provider_library": [provider_library(base_library, PROVIDER) for base_library in sorted(set(expected_base_libraries))],
    }
)

## Available Section Names

In [ ]:
pd.DataFrame(
    {
        "equity_historical": pd.Series(FMP_HISTORICAL_EQUITY_SECTIONS),
        "equity_extended": pd.Series(FMP_EXTENDED_EQUITY_SECTIONS),
        "etf": pd.Series(FMP_HISTORICAL_ETF_SECTIONS),
        "market_prices": pd.Series(MARKET_PRICE_SECTIONS.keys()),
        "equity_calendars": pd.Series(EQUITY_CALENDAR_SECTIONS),
    }
)